## Library Import

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from collections import Counter, deque
from tqdm import tqdm
import random
import copy
from PIL import Image
import open3d as o3d
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import time
from collections import defaultdict
from pprint import pprint


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
org_path="/mnt/Personal/Projects/Depth_Reconstruction/Test_Folder/stereo_test/"
# org_path="G:/Projects/Depth_Reconstruction/Test_Folder/stereo_test/"

object_name="Backpack"

test_input_path=org_path+"test_images/"+object_name+"/"
test_output_path=org_path+"test_outputs/"
test_3d_path=org_path+"test_3d/"

left_path = test_input_path+"im0.png"
right_path = test_input_path+"im1.png"
calib_path = test_input_path+"calib.txt"

left_truth_path = test_input_path+"disp0.pfm"
right_truth_path = test_input_path+"disp1.pfm"
# test_path= "/mnt/Personal/Projects/Depth_Reconstruction/Test_Folder/stereo_test/test_images/dustbin/im0.png"

In [ ]:
dataset_org_path = "/mnt/Extra/Project_Storage/stereo_ML_dataset/"

left_patch_memmap = dataset_org_path+"left_patch.dat"
right_strip_memmap = dataset_org_path+"right_strip.dat"
patch_disparity_memmap = dataset_org_path+"patch_disp.dat"

## Loading Functions

In [4]:
def load_calibration(file_path):
    
    # {
    #     'cam0': array([[2945.377,    0.   , 1284.862],
    #                 [   0.   , 2945.377,  954.52 ],
    #                 [   0.   ,    0.   ,    1.   ]]),
    #     'cam1': array([[2945.377,    0.   , 1455.543],
    #                 [   0.   , 2945.377,  954.52 ],
    #                 [   0.   ,    0.   ,    1.   ]]),
    #     'doffs': 170.681,
    #     'baseline': 178.232,
    #     'width': 2864,
    #     'height': 1924,
    #     'ndisp': 260,
    #     'isint': 0,
    #     'vmin': 32,
    #     'vmax': 224
    # }
        
    calib = {}
    
    with open(file_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
                
            # Split key and value
            if '=' in line:
                key, value = line.split('=', 1)
                key = key.strip()
                value = value.strip()
                
                # Handle matrix values (e.g., cam0=[...])
                if value.startswith('[') and value.endswith(']'):
                    matrix_str = value[1:-1]
                    # Split rows
                    rows = matrix_str.split(';')
                    matrix = []
                    for row in rows:
                        # Convert each row to float values
                        matrix.append([float(x) for x in row.split()])
                    value = np.array(matrix)
                
                # Handle scalar values
                else:
                    try:
                        # Try to convert to float if possible
                        value = float(value)
                        # Convert to int if no decimal part
                        if value.is_integer():
                            value = int(value)
                    except ValueError:
                        pass  # Keep as string if conversion fails
                
                calib[key] = value
                
    return calib

In [5]:

def load_image_to_rgb(image_path):
    """Load an image from path and return as RGB numpy array."""
    img = Image.open(image_path)
    return np.array(img.convert('RGB'))

In [6]:
def rgb_to_mono(img_array):
    """Convert RGB image array to luminance (grayscale) using standard weights."""
    if len(img_array.shape) == 2:
        return img_array  # Already grayscale
    gray = np.dot(img_array[..., :3], [0.33, 0.33, 0.33])
    return gray.astype(np.int_)

In [7]:

def load_pfm_disparity(file_path):
    
    try:
        with open(file_path, 'rb') as f:
            # Read header lines
            header = f.readline().decode('ascii').strip()
            dimensions = f.readline().decode('ascii').strip()
            scale = f.readline().decode('ascii').strip()
            
            # Parse dimensions
            width, height = map(int, dimensions.split())
            
            # Parse scale and determine endianness
            scale_factor = float(scale)
            little_endian = scale_factor < 0
            
            # Read binary data with correct endianness
            if little_endian:
                data = np.fromfile(f, dtype='<f4')  # Little-endian float32
            else:
                data = np.fromfile(f, dtype='>f4')  # Big-endian float32
            
            # Reshape the data
            expected_size = width * height
            if len(data) == expected_size:
                disparity_matrix = data.reshape((height, width))
            elif len(data) == expected_size * 3:
                # Color image - take mean of RGB channels
                disparity_matrix = data.reshape((height, width, 3)).mean(axis=2)
            else:
                raise ValueError(f"Unexpected data size: got {len(data)}, expected {expected_size}")
            
            disparity_matrix = disparity_matrix[::-1]
            
            disparity_matrix = np.where(disparity_matrix==np.inf,-1,disparity_matrix)
            
            return disparity_matrix
            
    except Exception as e:
        print(f"Error reading PFM file: {e}")
        raise

In [8]:
import numpy as np

def crop_array(array, target_shape):
    
    target_height, target_width = target_shape
    
    arr_shape = array.shape
    
    current_height, current_width = arr_shape[0],arr_shape[1]
    
    start_y = (current_height - target_height) // 2
    start_x = (current_width - target_width) // 2
    end_y = start_y + target_height
    end_x = start_x + target_width
    
    cropped_array = array[start_y:end_y, start_x:end_x]
    
    return cropped_array

## PLotting Functions

In [9]:

def display_image_array(img_array):
    
    # print(img_array.shape)

    
    """Display a numpy image array (2D or 3D) without axes."""
    plt.figure()
    if len(img_array.shape) == 3:  # RGB image
        plt.imshow(img_array)
    else:  # Grayscale
        plt.imshow(img_array, cmap='gray')
    plt.axis('off')
    plt.show()

In [10]:


def resize_image_array(image_array, scale_factor):
    # Convert array to PIL Image
    if len(image_array.shape) == 2:
        # Grayscale image
        img = Image.fromarray(image_array)
    elif len(image_array.shape) == 3:
        # RGB/RGBA image
        img = Image.fromarray(image_array.astype('uint8'))
    else:
        raise ValueError("Input array must be 2D (grayscale) or 3D (color)")
    
    # Calculate new dimensions
    width, height = img.size
    new_width = int(width * scale_factor)
    new_height = int(height * scale_factor)
    
    # Resize using Lanczos resampling (high quality)
    resized_img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
    
    # Convert back to numpy array
    resized_array = np.array(resized_img)
    
    # Preserve original dtype for grayscale
    if len(image_array.shape) == 2:
        resized_array = resized_array.astype(image_array.dtype)
    
    return resized_array

In [11]:
def plot_disp_array(data_array,colourbar=False):
    
    # print(data_array.shape)
    
    plt.figure(figsize=(6, 6))
    img = plt.imshow(data_array, cmap='viridis')
    plt.axis('off')
    if colourbar:
        cbar = plt.colorbar(img, fraction=0.046, pad=0.04)
        cbar.set_label('Value Scale', rotation=270, labelpad=15)
    
    plt.tight_layout()
    plt.show()

In [12]:
def plot_viridis_matrix(matrix):
    # Visualize results
    plt.figure(figsize=(12, 5))
    plt.imshow(matrix, cmap='viridis')
    plt.title('Matrix')
    plt.colorbar()

In [13]:

def plot_colorized_segments(array_2d):
    
    # Calculate value occurrences (excluding zeros from frequency calculation)
    flat_array = array_2d.flatten()
    value_counts = Counter(flat_array)
    unique_values = np.array(sorted(value_counts.keys()))
    
    # Separate zero and non-zero values
    zero_exists = 0 in value_counts
    non_zero_values = unique_values[unique_values != 0] if zero_exists else unique_values
    
    # Create Viridis colormap for non-zero values
    if len(non_zero_values) > 0:
        non_zero_counts = np.array([value_counts[v] for v in non_zero_values])
        norm_counts = (non_zero_counts - non_zero_counts.min()) / (non_zero_counts.max() - non_zero_counts.min() + 1e-10)
        cmap = plt.cm.viridis # type: ignore
        
        # Assign colors (using 0.1-0.9 range of Viridis to avoid extremes)
        color_dict = {val: cmap(0.1 + 0.8*norm_counts[i]) 
                     for i, val in enumerate(non_zero_values)}
    
    # Always set 0 to black
    if zero_exists:
        color_dict[0] = (0, 0, 0, 1)  # Black with full opacity
    
    # Create RGB image
    rgb_image = np.zeros((*array_2d.shape, 3))
    for val in color_dict:
        rgb_image[array_2d == val] = color_dict[val][:3]  # Exclude alpha channel
    
    # Plot
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb_image)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    # return color_dict  # Optional: return the color mapping

In [14]:
def display_six_arrays_with_cbar(img1, img2, img3, img4, disp1, disp2, 
                                show_colorbar=False, figsize=(15, 10)):
    """
    Display 6 arrays in a 3x2 grid layout with optional colorbars for disparity maps.
    
    Parameters:
    img1, img2, img3, img4: Image arrays (2D or 3D RGB)
    disp1, disp2: Disparity arrays (2D, displayed with viridis colormap)
    show_colorbar: Whether to show colorbars for disparity maps
    figsize: Tuple specifying the figure size (width, height)
    """
    if show_colorbar:
        fig, axes = plt.subplots(3, 2, figsize=(16, 12))
    else:
        fig, axes = plt.subplots(3, 2, figsize=figsize)
    
    # First row: img1 and img2
    if len(img1.shape) == 3:
        axes[0, 0].imshow(img1)
    else:
        axes[0, 0].imshow(img1, cmap='gray')
    axes[0, 0].axis('off')
    
    if len(img2.shape) == 3:
        axes[0, 1].imshow(img2)
    else:
        axes[0, 1].imshow(img2, cmap='gray')
    axes[0, 1].axis('off')
    
    # Second row: img3 and img4
    if len(img3.shape) == 3:
        axes[1, 0].imshow(img3)
    else:
        axes[1, 0].imshow(img3, cmap='gray')
    axes[1, 0].axis('off')
    
    if len(img4.shape) == 3:
        axes[1, 1].imshow(img4)
    else:
        axes[1, 1].imshow(img4, cmap='gray')
    axes[1, 1].axis('off')
    
    # Third row: disp1 and disp2
    im1 = axes[2, 0].imshow(disp1, cmap='viridis')
    axes[2, 0].axis('off')
    if show_colorbar:
        plt.colorbar(im1, ax=axes[2, 0], fraction=0.046, pad=0.04)
    
    im2 = axes[2, 1].imshow(disp2, cmap='viridis')
    axes[2, 1].axis('off')
    if show_colorbar:
        plt.colorbar(im2, ax=axes[2, 1], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    plt.show()

In [15]:
import numpy as np
import matplotlib.pyplot as plt

def plot_edge_matrix(matrix):
    
    rows, cols = matrix.shape
    edge_matrix = np.full(matrix.shape,fill_value=1,dtype=int)
    
    # Define 4-adjacency offsets (up, down, left, right)
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    for i in range(rows):
        for j in range(cols):
            current_val = matrix[i, j]
            is_edge = False
            
            # Check all 4 adjacent neighbors
            for di, dj in directions:
                ni, nj = i + di, j + dj
                if 0 <= ni < rows and 0 <= nj < cols:  # Check bounds
                    if matrix[ni, nj] != current_val and edge_matrix[ni, nj] != 0:
                        is_edge = True
                        break
            
            if is_edge:
                edge_matrix[i, j] = 0
                
    
    # Plot the result
    plt.figure(figsize=(10, 10))
    plt.imshow(edge_matrix, cmap='binary', vmin=0, vmax=1)  # 0=white, 1=black
    plt.axis('off')
    plt.show()

## Loading Data

In [16]:
import os

def get_subfolders(folder_path):
    
    try:
        # Check if the path exists and is a directory
        if not os.path.exists(folder_path):
            raise FileNotFoundError(f"Folder not found: {folder_path}")
        
        if not os.path.isdir(folder_path):
            raise NotADirectoryError(f"Path is not a directory: {folder_path}")
        
        # Get all items in the directory
        all_items = os.listdir(folder_path)
        
        # Filter only directories
        subfolders = {item:folder_path+item+"/" for item in all_items 
                    if os.path.isdir(os.path.join(folder_path, item))}
        
        
        return subfolders
        
    except Exception as e:
        print(f"Error: {e}")
        return []

# Example usage:
test_folders = get_subfolders(org_path+"test_images/")
# pprint(test_folders)
print(f"Test Files - {len(test_folders)}")

Test Files - 23


In [17]:

calib_list={}
rgb_image_list={}
gray_image_list={}
gray_image_flipped_list={}
truth_image_list={}
truth_image_flipped_list={}

resize_fraction = 1
resize_factor = 1/resize_fraction


In [18]:


def load_data_from_files(folders):

    for folder in tqdm(folders):
        
        calib = load_calibration(folders[folder]+"calib.txt")
        
        rgb_l = load_image_to_rgb(folders[folder]+"im0.png")
        rgb_r = load_image_to_rgb(folders[folder]+"im1.png")
        
        rgb_l = resize_image_array(rgb_l,resize_factor)
        rgb_r = resize_image_array(rgb_r,resize_factor)
        
        gray_l = rgb_to_mono(rgb_l)
        gray_r = rgb_to_mono(rgb_r)
        
        truth_l = load_pfm_disparity(folders[folder]+"disp0.pfm")
        truth_r = load_pfm_disparity(folders[folder]+"disp1.pfm")
        
        truth_l = resize_image_array(truth_l,resize_factor)
        truth_l = truth_l//resize_fraction
        truth_r = resize_image_array(truth_r,resize_factor)
        truth_r = truth_r//resize_fraction
        
        calib_list[folder] = calib
        
        rgb_image_list[folder]={'left':rgb_l,'right':rgb_r}
        
        gray_image_list[folder]={'left':gray_l,'right':gray_r}
        gray_image_flipped_list[folder]={'left':gray_l,'right':gray_r}
        
        truth_image_list[folder]={'left':truth_l,'right':truth_r}
        truth_image_flipped_list[folder]={'left':truth_l,'right':truth_r}
        
    min_shape = [np.inf,np.inf]
    
    for folder in folders:
        
        org_shape = gray_image_list[folder]['left'].shape
        
        if org_shape[0]<min_shape[0]:
            min_shape[0] = org_shape[0]
            
        if org_shape[1]<min_shape[1]:
            min_shape[1] = org_shape[1]
            
    for folder in tqdm(folders):
        
        rgb_image_list[folder]['left'] = crop_array(rgb_image_list[folder]['left'],min_shape)
        rgb_image_list[folder]['right'] = crop_array(rgb_image_list[folder]['right'],min_shape)
        
        gray_image_list[folder]['left'] = crop_array(gray_image_list[folder]['left'],min_shape)
        gray_image_list[folder]['right'] = crop_array(gray_image_list[folder]['right'],min_shape)
        
        gray_image_flipped_list[folder]['left'] = crop_array(gray_image_flipped_list[folder]['left'],min_shape)
        gray_image_flipped_list[folder]['right'] = crop_array(gray_image_flipped_list[folder]['right'],min_shape)
        
        gray_image_flipped_list[folder]['left'] = np.fliplr(gray_image_flipped_list[folder]['left'])
        gray_image_flipped_list[folder]['right'] = np.fliplr(gray_image_flipped_list[folder]['right'])
        
        truth_image_list[folder]['left'] = crop_array(truth_image_list[folder]['left'],min_shape)
        truth_image_list[folder]['right'] = crop_array(truth_image_list[folder]['right'],min_shape)
        
        truth_image_flipped_list[folder]['left'] = crop_array(truth_image_flipped_list[folder]['left'],min_shape)
        truth_image_flipped_list[folder]['right'] = crop_array(truth_image_flipped_list[folder]['right'],min_shape)
        
        truth_image_flipped_list[folder]['left'] = np.fliplr(truth_image_flipped_list[folder]['left'])
        truth_image_flipped_list[folder]['right'] = np.fliplr(truth_image_flipped_list[folder]['right'])
        
load_data_from_files(test_folders)


100%|██████████| 23/23 [00:00<00:00, 67413.69it/s]


In [19]:
def folder_data_printer(name):
    
    calib = calib_list[name]
    rgb = rgb_image_list[name]
    gray = gray_image_list[name]
    truth = truth_image_list[name]
    
    print(gray['left'].shape,"\n\n")
    pprint(calib)
    
    # display_six_arrays_with_cbar(rgb['left'], rgb['right'], gray['left'], gray['right'], truth['left'], truth['right'], figsize=(10, 10))
    
    # display_image_array(rgb['left'])
    # display_image_array(rgb['right'])
    # display_image_array(gray['left'])
    # display_image_array(gray['right'])
    
    # plot_disp_array(truth['left'])
    # plot_disp_array(truth['right'])
    
    return np.array(gray['left'].shape)
    
global_image_shape = folder_data_printer('Shopvac')

## ndisp tight bound - 100

(1848, 2300) 


{'baseline': 379.965,
 'cam0': array([[7.242753e+03, 0.000000e+00, 1.079538e+03],
       [0.000000e+00, 7.242753e+03, 1.018846e+03],
       [0.000000e+00, 0.000000e+00, 1.000000e+00]]),
 'cam1': array([[7.242753e+03, 0.000000e+00, 1.588865e+03],
       [0.000000e+00, 7.242753e+03, 1.018846e+03],
       [0.000000e+00, 0.000000e+00, 1.000000e+00]]),
 'doffs': 509.327,
 'dyavg': 0,
 'dymax': 0,
 'height': 1996,
 'isint': 0,
 'ndisp': 1110,
 'vmax': 1100,
 'vmin': 7,
 'width': 2356}


## Patch Generation

In [ ]:
def generate_patches_strips(patch_shape, target, disp_threshold_difference, disp_threshold_percent, max_disp=100, maximum_tries=1000,flush_interval=10000):
    
    folder_list = [folder for folder in test_folders]
    
    patch_list = np.memmap(left_patch_memmap, dtype=np.uint8, mode='w+', shape=(target, patch_shape[0], patch_shape[1]))
    strip_list = np.memmap(right_strip_memmap, dtype=np.uint8, mode='w+', shape=(target, patch_shape[0], patch_shape[1]+max_disp))
    disp_list = np.memmap(patch_disparity_memmap, dtype=np.int16, mode='w+', shape=(target,))
    
    
    pbar = tqdm(range(target), desc="Generating patches")
    
    for i in pbar:
        
        try_count = 0
        patch_found = False
        
        choices = ['left','right']
        side_choice = random.randint(0, 1)
        
        side = choices[side_choice]
        other_side = choices[1-side_choice]
        
        while try_count < maximum_tries and not patch_found:
            
            # # Update the progress bar description with current try count
            # if try_count%20==0:
            #     pbar.set_description(f"Patch {i}/{target} - Try {try_count+1}/{maximum_tries}")
            
            folder_name = np.random.choice(folder_list)
            
            if side=='left':
                
                disp_patch = truth_image_list[folder_name][side]
                disp_strip = truth_image_list[folder_name][other_side]
                
                patch_image = gray_image_list[folder_name][side]
                strip_image = gray_image_list[folder_name][other_side]
            
            elif side == 'right':
                
                disp_patch = truth_image_flipped_list[folder_name][side]
                disp_strip = truth_image_flipped_list[folder_name][other_side]
                
                patch_image = gray_image_flipped_list[folder_name][side]
                strip_image = gray_image_flipped_list[folder_name][other_side]
                
            
            image_height, image_width = disp_patch.shape
            
            y_min = np.random.randint(0, image_height - patch_shape[0])
            x_min = np.random.randint(max_disp, image_width - patch_shape[1])
            y_max = y_min + patch_shape[0]
            x_max = x_min + patch_shape[1]
            
            patch_mask = disp_patch[y_min:y_max, x_min:x_max] 
            valid_patch_mask = np.isfinite(patch_mask)
            valid_patch_values = patch_mask[valid_patch_mask]
            
            if np.sum(valid_patch_mask)>0:
                median_patch_disp = np.median(valid_patch_values)
            else:
                median_patch_disp=-1
            
            if 0 < median_patch_disp < max_disp:
            
                strip_mask = disp_strip[y_min:y_max, (x_min-max_disp):x_max]            
                valid_strip_mask = np.isfinite(strip_mask)
                valid_strip_values = strip_mask[valid_strip_mask]
                median_strip_disp = np.median(valid_strip_values)
                
                if np.sum(valid_patch_mask) > 0 and np.sum(valid_strip_mask) > 0:  
            
                    thresh_percent = np.mean(np.abs(valid_strip_values - median_strip_disp) >= disp_threshold_difference)
                    
                    if thresh_percent >= disp_threshold_percent:
                        
                        image_patch = patch_image[y_min:y_max, x_min:x_max]
                        
                        image_strip = strip_image[y_min:y_max, (x_min-max_disp):x_max] 
                        
                        patch_list[i] = image_patch
                        strip_list[i] = image_strip
                        disp_list[i] = median_patch_disp
                        
                        patch_found = True
            
            try_count += 1
        
        if not patch_found:
            print(f"Warning: Could not find suitable patch for index {i} after {maximum_tries} tries")
            
            patch_list[i] = np.zeros(patch_shape, dtype=np.uint8)
            disp_list[i] = 0
            
            break
        
        if (i + 1) % flush_interval == 0:
            patch_list.flush()
            disp_list.flush()
            strip_list.flush()
    
    patch_list.flush()
    disp_list.flush()
    strip_list.flush()
    
    return patch_list, disp_list, strip_list


maximum_disparity = int(800*resize_factor)
disparity_threshold = maximum_disparity*0.3

strip_outside_thresh_percent = 0.3

# global_patch_shape = (min(global_image_shape//30),min(global_image_shape//30))
global_patch_shape = (50,50)

patch_list, disp_list, strip_list = generate_patches_strips(global_patch_shape,target=500000, \
    max_disp=maximum_disparity,disp_threshold_difference=disparity_threshold,disp_threshold_percent=strip_outside_thresh_percent,maximum_tries=100000)

Generating patches: 100%|██████████| 500000/500000 [1:41:53<00:00, 81.79it/s]  


In [21]:
print(patch_list.shape)
print(disp_list.shape)
print(strip_list.shape)

# for i in range(0,10):
    
#     image_array = patch_list[i]
#     strip_arr = strip_list[i]
#     display_image_array(image_array)
#     display_image_array(strip_arr)
    
#     print(disp_list[i])
    

(500000, 50, 50)
(500000,)
(500000, 50, 850)


In [22]:
import os
import numpy as np
import math
import shutil

def split_memmaps(org_folder, train_folder, test_folder,
                patch_shape, max_disp, target,train_split=0.8,
                chunk_size=10000):


    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)

    # Split indices
    train_count = math.floor(target * train_split)
    test_count = target - train_count

    print(f"Splitting {target} samples → {train_count} train / {test_count} test")

    # File names you expect in org_folder
    file_triplets = [
        ("left_patch.dat", np.uint8, (target, patch_shape[0], patch_shape[1])),
        ("patch_disp.dat", np.int16, (target,)),
        ("right_strip.dat", np.uint8, (target, patch_shape[0], patch_shape[1] + max_disp)),
    ]

    for fname, dtype, shape in file_triplets:
        src_path = os.path.join(org_folder, fname)
        if not os.path.exists(src_path):
            print(f"Skipping missing file: {fname}")
            continue

        print(f"Processing {fname} ...")

        # Open source memmap
        src = np.memmap(src_path, dtype=dtype, mode='r', shape=shape)

        # Create destination memmaps
        train_shape = (train_count,) + shape[1:]
        test_shape = (test_count,) + shape[1:]

        train_dest = np.memmap(os.path.join(train_folder, fname), dtype=dtype, mode='w+', shape=train_shape)
        test_dest = np.memmap(os.path.join(test_folder, fname), dtype=dtype, mode='w+', shape=test_shape)

        # Copy in chunks
        start = 0
        while start < target:
            end = min(start + chunk_size, target)
            chunk = src[start:end]

            if end <= train_count:
                train_dest[start:end] = chunk
            elif start >= train_count:
                test_dest[start - train_count:end - train_count] = chunk
            else:
                # boundary split
                split = train_count - start
                train_dest[start:train_count] = chunk[:split]
                test_dest[0:end - train_count] = chunk[split:]

            start = end

        # Flush and cleanup
        del src, train_dest, test_dest
        print(f"Done splitting {fname}")

    print("All files processed successfully.")
    
split_memmaps(
    org_folder="//mnt/Extra/Project_Storage/stereo_ML_dataset/org/",
    train_folder="/mnt/Extra/Project_Storage/stereo_ML_dataset/train",
    test_folder="/mnt/Extra/Project_Storage/stereo_ML_dataset/test/",
    patch_shape=(50, 50),
    max_disp=800,
    target=500000,
    chunk_size=10000  # adjustable for memory
)



Splitting 500000 samples → 400000 train / 100000 test


  0%|          | 0/3 [00:00<?, ?it/s]

Processing left_patch.dat ...


 33%|███▎      | 1/3 [00:02<00:05,  2.67s/it]

Done splitting left_patch.dat
Processing patch_disp.dat ...
Done splitting patch_disp.dat
Processing right_strip.dat ...


100%|██████████| 3/3 [02:49<00:00, 62.30s/it]

Done splitting right_strip.dat


100%|██████████| 3/3 [02:49<00:00, 56.57s/it]


All files processed successfully.
